# 01 — Raw HOG Features

Loads all patient GIF images from the OASIS-1 patient folders, extracts HOG features,
averages the 5 image vectors per patient, and saves the result as a CSV.

**Output:** `raw_hog_features.csv` — 416 rows × 8,101 columns (`patient_id` + 8,100 HOG features)

In [1]:
import glob
import os

import numpy as np
import pandas as pd
from PIL import Image
from skimage.feature import hog
from skimage.transform import resize

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
OUTPUT_CSV = os.path.join(DATA_DIR, "notebooks", "raw_hog_features.csv")

# ── HOG parameters ────────────────────────────────────────────────────────────
TARGET_SIZE      = (128, 128)
ORIENTATIONS     = 9
PIXELS_PER_CELL  = (8, 8)
CELLS_PER_BLOCK  = (2, 2)

# Expected feature length: 15×15 blocks × 2×2 cells × 9 orientations = 8,100
HOG_LEN = 8_100

In [3]:
def load_image(path):
    img = Image.open(path).convert("L")
    arr = np.array(img, dtype=np.float32) / 255.0

    # Pad to square using the larger dimension (preserves aspect ratio)
    h, w = arr.shape
    side  = max(h, w)
    pad   = np.zeros((side, side), dtype=np.float32)
    row_off = (side - h) // 2
    col_off = (side - w) // 2
    pad[row_off:row_off + h, col_off:col_off + w] = arr

    return resize(pad, TARGET_SIZE, anti_aliasing=True)


def extract_hog(arr):
    """Return the HOG feature vector for a 2-D greyscale array."""
    features, _ = hog(
        arr,
        orientations=ORIENTATIONS,
        pixels_per_cell=PIXELS_PER_CELL,
        cells_per_block=CELLS_PER_BLOCK,
        visualize=True,
        feature_vector=True,
    )
    return features

In [4]:
patient_dirs = sorted(
    d for d in glob.glob(os.path.join(DATA_DIR, "OAS1_*"))
    if os.path.isdir(d)
)
print(f"Found {len(patient_dirs)} patient folders")

Found 416 patient folders


In [5]:
patient_ids   = []
patient_index = []   # maps each image row → patient index
hog_rows      = []

for p_idx, p_dir in enumerate(patient_dirs):
    pid       = os.path.basename(p_dir)
    gif_paths = sorted(glob.glob(os.path.join(p_dir, "*.gif")))

    if not gif_paths:
        print(f"  WARNING: no GIFs in {pid}, skipping")
        continue

    patient_ids.append(pid)
    for gpath in gif_paths:
        hog_rows.append(extract_hog(load_image(gpath)))
        patient_index.append(len(patient_ids) - 1)

    if (p_idx + 1) % 50 == 0 or (p_idx + 1) == len(patient_dirs):
        print(f"  Processed {p_idx + 1}/{len(patient_dirs)} patients")

hog_matrix    = np.array(hog_rows, dtype=np.float32)
patient_index = np.array(patient_index)

print(f"\nHOG matrix shape : {hog_matrix.shape}")
print(f"Feature length   : {hog_matrix.shape[1]}  (expected {HOG_LEN})")

  Processed 50/416 patients
  Processed 100/416 patients
  Processed 150/416 patients
  Processed 200/416 patients
  Processed 250/416 patients
  Processed 300/416 patients
  Processed 350/416 patients
  Processed 400/416 patients
  Processed 416/416 patients

HOG matrix shape : (2080, 8100)
Feature length   : 8100  (expected 8100)


In [6]:
n_patients      = len(patient_ids)
n_features      = hog_matrix.shape[1]
patient_vectors = np.zeros((n_patients, n_features), dtype=np.float32)

for p_idx in range(n_patients):
    mask = patient_index == p_idx
    patient_vectors[p_idx] = hog_matrix[mask].mean(axis=0)

print(f"Per-patient averaged matrix shape: {patient_vectors.shape}")
print(f"  → {n_patients} patients × {n_features} HOG features")

Per-patient averaged matrix shape: (416, 8100)
  → 416 patients × 8100 HOG features


In [7]:
col_names = [f"HOG_{i}" for i in range(n_features)]
df = pd.DataFrame(patient_vectors, columns=col_names)
df.insert(0, "patient_id", patient_ids)

df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved  : {OUTPUT_CSV}")
print(f"Shape  : {df.shape}  (rows=patients, cols=patient_id + HOG features)")
print()
print(df.iloc[:3, :6].to_string())

Saved  : C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\raw_hog_features.csv
Shape  : (416, 8101)  (rows=patients, cols=patient_id + HOG features)

  patient_id  HOG_0  HOG_1  HOG_2  HOG_3  HOG_4
0  OAS1_0001    0.0    0.0    0.0    0.0    0.0
1  OAS1_0002    0.0    0.0    0.0    0.0    0.0
2  OAS1_0003    0.0    0.0    0.0    0.0    0.0
